In [50]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic pdfplumber faiss-cpu pdfplumber tiktoken tqdm python-dotenv pylance pypdf nest_asyncio



Note: you may need to restart the kernel to use updated packages.


In [51]:
%pip install "crewai[azure-ai-inference]"

Note: you may need to restart the kernel to use updated packages.


In [52]:
%pip install litellm

Note: you may need to restart the kernel to use updated packages.


In [53]:
# ============================================================================
# SECTION 1: ENVIRONMENT SETUP & IMPORTS
# ============================================================================

import os
import json
import logging
from datetime import datetime
from typing import List, Dict, Any
from pathlib import Path
import requests

# Import CrewAI
from crewai import Agent, Task, Crew
from crewai_tools import PDFSearchTool
from crewai.tools import BaseTool
from crewai.llm import LLM

# Import LangChain
from langchain_core.tools import tool, Tool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Import Tavily
from tavily import TavilyClient

# Configure logging for clear visibility
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ All imports successful")

✓ All imports successful


In [54]:
# ============================================================================
# SECTION 2: ENVIRONMENT CONFIGURATION
# ============================================================================

def setup_environment():
    """Load environment variables and configure API clients"""
    load_dotenv()
    
    openai_api_key = os.getenv("OPENAI_API_KEY", "")
    tavily_api_key = os.getenv("TAVILY_API_KEY", "")
    
    if not openai_api_key:
        raise ValueError("⚠️ OPENAI_API_KEY not set in .env file")
    if not tavily_api_key:
        raise ValueError("⚠️ TAVILY_API_KEY not set in .env file")
    
    logger.info("✓ Environment variables loaded successfully")
    
    return openai_api_key, tavily_api_key

# Load environment
try:
    openai_key, tavily_key = setup_environment()
    print("✓ API keys configured")
except ValueError as e:
    print(f"Error: {str(e)}")
    print("\nPlease ensure .env file exists with:")
    print("  OPENAI_API_KEY=<your-key>")
    print("  TAVILY_API_KEY=<your-key>")

2026-05-23 15:14:44,967 - __main__ - INFO - ✓ Environment variables loaded successfully


✓ API keys configured


In [55]:
# ============================================================================
# SECTION 3: LLM & CLIENT INITIALIZATION
# ============================================================================

def initialize_llm(openai_api_key: str):
    """Initialize the language model"""
    llm = LLM(
        model="azure/gpt-5-mini",
        base_url="https://openai-api-management-gw.azure-api.net/",
        temperature=0.7,
        api_key=openai_api_key,
        is_litellm=True
    )
    logger.info("✓ LLM initialized (GPT-5-mini)")
    return llm

def initialize_tavily(tavily_api_key: str):
    """Initialize Tavily search client"""
    client = TavilyClient(api_key=tavily_api_key)
    logger.info("✓ Tavily client initialized")
    return client

# Initialize components
llm = initialize_llm(openai_key)
tavily_client = initialize_tavily(tavily_key)

print("✓ LLM and Tavily client ready")

2026-05-23 15:14:44,989 - __main__ - INFO - ✓ LLM initialized (GPT-5-mini)
2026-05-23 15:14:44,990 - __main__ - INFO - ✓ Tavily client initialized


✓ LLM and Tavily client ready


In [56]:
# ============================================================================
# SECTION 4: TOOL DEFINITIONS
# ============================================================================


# ---- Custom CrewAI Tool for Web Search ----
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."
    def _run(self, query: str):
        url = "https://api.tavily.com/search"
        
        payload = {
            "api_key": os.getenv("TAVILY_API_KEY", ""),
            "query": query,
            "max_results": 5
        }
        response = requests.post(url, json=payload)
        data = response.json()
    
        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")
    
        return "\n".join(results)

search_tool = TavilySearchTool()  

print("✓ Web search Tools ready")

✓ Web search Tools ready


In [57]:
def create_pdf_search_tool():
    """Create PDF search tool for document retrieval"""
    
    try:
        # Initialize Azure Chat OpenAI
        azure_llm = AzureChatOpenAI(
            azure_endpoint="https://openai-api-management-gw.azure-api.net",
            api_version="2025-01-01-preview",
            deployment_name="gpt-5-mini"
        )
        
        pdf_tool = PDFSearchTool(
            pdf='documents/Lesson02Demo01.pdf',
            llm=azure_llm
        )
        logger.info("✓ PDF search tool initialized with Azure LLM")
    except Exception as e:
        logger.warning(f"PDF tool initialization note: {str(e)}")
        pdf_tool = None
    
    def pdf_search(query: str) -> str:
        """Search PDF documents for information"""
        if not pdf_tool:
            return " PDF search not available (pdf not found)"
        try:
            results = pdf_tool.run(query)
            return f" PDF Search Results for '{query}':\n{results}"
        except Exception as e:
            return f"Error during PDF search: {str(e)}"
    
    return Tool(
        name="pdf_search",
        func=pdf_search,
        description="Search PDF documents for domain-specific information"
    )

pdf_search_tool=create_pdf_search_tool()

2026-05-23 15:14:45,395 - __main__ - WARNING - PDF tool initialization note: Error code: 401 - {'error': {'message': 'Incorrect API key provided: 2ABecnfx************************************************************************i3sC. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}} in upsert.


In [58]:
# ============================================================================
# SECTION 5: AGENT DEFINITIONS
# ============================================================================

def create_router_agent(llm):
    """
    Create Router Agent
    
    Responsibility: Analyze user questions and determine optimal retrieval path
    - Understands semantic content of queries
    - Considers temporal context (current vs. historical)
    - Routes to: PDF Search, Web Search, or Direct LLM
    """
    
    return Agent(
        role="Query Router and Analyzer",
        goal="""Analyze user questions and intelligently determine the best retrieval path:
        • PDF Search: For domain-specific, internal documentation
        • Web Search: For current events and real-time information
        • Direct LLM: For general knowledge that doesn't require retrieval
        
        Provide clear reasoning for your routing decision.""",
        backstory="""You are an expert question classifier with deep knowledge of 
        information sources and retrieval strategies. You make optimal routing decisions 
        based on query type, temporal context, and information freshness requirements.""",
        llm=llm,
        verbose=True,
        allow_delegation=False
    )

def create_retriever_agent(llm, search_tool):
    """
    Create Retriever Agent
    
    Responsibility: Execute retrieval and synthesize comprehensive answers
    - Executes tools selected by Router
    - Synthesizes information from multiple sources
    - Manages citations and source attribution
    - Formats answers for clarity
    """
    
    return Agent(
        role="Information Retriever and Answer Synthesizer",
        goal="""Execute information retrieval from the Router Agent's recommended path and 
        synthesize accurate, well-sourced answers with proper citations and confidence levels.
        If retrieval fails, provide clear explanation and attempt alternative approaches.""",
        backstory="""You are an expert information retriever and synthesizer with excellence 
        in finding relevant information and providing comprehensive answers with proper source 
        attribution. You ensure all claims are grounded in retrieved data.""",
        tools=[search_tool],
        llm=llm,
        verbose=True,
        allow_delegation=False
    )

# Create agents
router_agent = create_router_agent(llm)
retriever_agent = create_retriever_agent(llm, search_tool)

print("✓ Router and Retriever agents created")

✓ Router and Retriever agents created


In [59]:
# ============================================================================
# SECTION 6: INTERACTION TRACING & LOGGING
# ============================================================================

class InteractionTracer:
    """
    Track and log all agent interactions for reasoning traceability
    
    Provides:
    - Complete interaction history
    - Timestamps for all events
    - Agent actions and decisions
    - Error tracking
    - Execution duration
    """
    
    def __init__(self):
        self.interactions = []
        self.start_time = None
        self.end_time = None
    
    def start_trace(self, user_question: str):
        """Initialize trace for new query"""
        self.start_time = datetime.now()
        self.interactions = []
        self.log_interaction(
            event_type='START',
            agent='System',
            action='Query received',
            details={'user_question': user_question}
        )
    
    def log_interaction(self, event_type: str, agent: str, action: str, details: Dict = None):
        """Log an interaction event"""
        interaction = {
            'timestamp': datetime.now().isoformat(),
            'event_type': event_type,
            'agent': agent,
            'action': action,
            'details': details or {}
        }
        self.interactions.append(interaction)
        logger.info(f"[{agent}] {action}")
    
    def end_trace(self, final_answer: str):
        """Mark end of trace"""
        self.end_time = datetime.now()
        duration = (self.end_time - self.start_time).total_seconds()
        self.log_interaction(
            event_type='END',
            agent='System',
            action='Query completed',
            details={'duration_seconds': round(duration, 2)}
        )
    
    def get_summary(self) -> Dict:
        """Get complete trace summary"""
        return {
            'total_interactions': len(self.interactions),
            'duration_seconds': (self.end_time - self.start_time).total_seconds() if self.end_time else None,
            'interactions': self.interactions
        }
    
    def display_trace(self):
        """Pretty print trace for visualization"""
        print("\n" + "="*80)
        print("REASONING TRACE - Agent Interactions")
        print("="*80 + "\n")
        
        for i, interaction in enumerate(self.interactions, 1):
            timestamp = interaction['timestamp'].split('T')[1].split('.')[0]
            event = interaction['event_type']
            agent = interaction['agent']
            action = interaction['action']
            
            print(f"{i:2d}. [{timestamp}] [{event:6s}] {agent:20s} → {action}")
            
            if interaction['details']:
                for key, value in interaction['details'].items():
                    if isinstance(value, str) and len(value) > 60:
                        print(f"     └─ {key}: {value[:60]}...")
                    else:
                        print(f"     └─ {key}: {value}")
        
        print("\n" + "="*80)
        summary = self.get_summary()
        print(f"Total Interactions: {summary['total_interactions']} | Duration: {summary['duration_seconds']:.2f}s")
        print("="*80 + "\n")

# Create tracer instance
tracer = InteractionTracer()
print("✓ Interaction tracer initialized")

✓ Interaction tracer initialized


In [60]:
# ============================================================================
# SECTION 7: CREW ORCHESTRATION
# ============================================================================

def create_crew(router_agent, retriever_agent):
    """
    Create and configure the CrewAI Crew
    
    The crew orchestrates two sequential tasks:
    1. Router Task: Analyze and classify the query
    2. Retrieval Task: Execute retrieval and synthesize answer
    
    Process: Sequential (one task after another)
    """
    
    # Define routing task
    routing_task = Task(
        description="""Analyze the following user question and determine the best retrieval path:
        
        Question: {user_question}
        
        Provide:
        1. Analysis of the question type and temporal context
        2. Routing decision (PDF Search, Web Search, or Direct LLM)
        3. Detailed reasoning for your choice""",
        expected_output="""Clear routing decision with reasoning, e.g.:
        'ROUTE: Web Search - The question asks about current events which requires real-time information'""",
        agent=router_agent,
        async_execution=False
    )
    
    # Define retrieval task
    retrieval_task = Task(
        description="""Based on the routing analysis from the previous agent, 
        retrieve information to answer this question: {user_question}
        
        Then synthesize a comprehensive answer with:
        1. Direct answer to the question
        2. Supporting details from retrieved sources
        3. Proper source attribution and citations
        4. Confidence level assessment""",
        expected_output="""Well-researched answer with:
        - Clear answer statement
        - Supporting evidence from sources
        - Full source citations with URLs
        - Confidence assessment""",
        agent=retriever_agent,
        context=[routing_task],
        async_execution=False
    )
    # Create crew with sequential process
    crew = Crew(
        agents=[router_agent, retriever_agent],
        tasks=[routing_task, retrieval_task],
        verbose=True,
        process='sequential'
    )
    
    logger.info("✓ Crew created with sequential process")
    return crew

# Initialize the crew
crew = create_crew(router_agent, retriever_agent)
print("✓ Crew orchestration ready")

2026-05-23 15:14:45,457 - __main__ - INFO - ✓ Crew created with sequential process


✓ Crew orchestration ready


In [61]:
# ============================================================================
# SECTION 8: QUERY EXECUTION ENGINE
# ============================================================================

import asyncio
from nest_asyncio import apply

# Apply nest_asyncio to allow async execution in Jupyter
apply()

def execute_query(crew, question: str, tracer: InteractionTracer) -> Dict:
    """
    Execute a user query through the complete RAG pipeline
    
    Steps:
    1. Log query start
    2. Router Agent analyzes and routes
    3. Retriever Agent executes and synthesizes
    4. Log results and trace
    
    Returns: Complete execution result with answer and trace
    """
    
    print(f"\n{'='*80}")
    print(f" QUERY: {question}")
    print(f"{'='*80}\n")
    
    tracer.start_trace(question)
    
    try:
          # Use async execution
        result = asyncio.run(crew.kickoff_async(inputs={"user_question": question}))
      
        # Mark trace completion
        tracer.end_trace(str(result))
        
        # Display answer
        print(f"\n{'='*80}")
        print("✅ ANSWER GENERATED")
        print(f"{'='*80}\n{result}\n")
        
        return {
            'question': question,
            'answer': str(result),
            'trace': tracer.get_summary(),
            'success': True
        }
    
    except Exception as e:
        # Handle errors
        error_msg = f"Execution Error: {str(e)}"
        logger.error(error_msg)
        
        tracer.log_interaction(
            event_type='ERROR',
            agent='System',
            action='Execution failed',
            details={'error': str(e)}
        )
        
        print(f"\n❌ Error: {error_msg}\n")
        
        return {
            'question': question,
            'answer': None,
            'error': error_msg,
            'trace': tracer.get_summary(),
            'success': False
        }

print("✓ Query execution engine ready")

✓ Query execution engine ready


In [62]:
# ============================================================================
# SECTION 9: TEST QUERY EXECUTION
# ============================================================================

# Define diverse test queries covering different routing scenarios
test_queries = [
    {
        'query': "What are the latest developments in generative AI?",
        'expected_route': 'Web Search',
        'reason': 'Requires current information'
    },
    {
        'query': "Explain the transformer architecture and attention mechanisms",
        'expected_route': 'Direct LLM or Web Search',
        'reason': 'General knowledge question'
    },
    {
        'query': "What are the key benefits of Python for data science?",
        'expected_route': 'Web Search or Direct LLM',
        'reason': 'General knowledge with potential current best practices'
    }
]

print("\n" + "="*80)
print(" EXECUTING TEST QUERIES")
print("="*80 + "\n")

results = []

for i, query_info in enumerate(test_queries, 1):
    question = query_info['query']
    
    print(f"\n{'─'*80}")
    print(f"Query {i}/{len(test_queries)}")
    print(f"Expected Route: {query_info['expected_route']}")
    print(f"Reason: {query_info['reason']}")
    print(f"{'─'*80}\n")
    
    # Reset tracer for new query
    tracer = InteractionTracer()
    
    # Execute query
    result = execute_query(crew, question, tracer)
    results.append(result)
    
    # Display trace
    tracer.display_trace()
    
    # Add status indicator
    status = "✅ SUCCESS" if result['success'] else "❌ FAILED"
    print(f"{status} - Query {i} completed\n")

print("\n" + "="*80)
print("✅ EXECUTION COMPLETE - All test queries processed")
print("="*80 + "\n")

2026-05-23 15:14:45,565 - __main__ - INFO - [System] Query received



 EXECUTING TEST QUERIES


────────────────────────────────────────────────────────────────────────────────
Query 1/3
Expected Route: Web Search
Reason: Requires current information
────────────────────────────────────────────────────────────────────────────────


 QUERY: What are the latest developments in generative AI?



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e097df75-afa8-4cb3-ac58-cf8e10c58e1b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following user question and determine the best retrieval path:                               │
│                                                                                                                 │
│          Question: What are the latest developments in generative AI?                                           │
│                                                                                                                 │
│          Provide:                                                                                               │
│          1. Analysis of the question type and temporal context                                                  │
│          2. Routing decision (PDF Search, Web Search, or Direct LLM)                                            │
│          3. Detailed reasoning for your choice                                                                  │
│  ID: a65c1498-d28d-479c-9dab-065334e03cf1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router and Analyzer                                                                               │
│                                                                                                                 │
│  Task: Analyze the following user question and determine the best retrieval path:                               │
│                                                                                                                 │
│          Question: What are the latest developments in generative AI?                                           │
│                                                                                                                 │
│          Provide:                                                                                               │
│          1. Analysis of the question type and temporal context                                                  │
│          2. Routing decision (PDF Search, Web Search, or Direct LLM)                                            │
│          3. Detailed reasoning for your choice                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the routing analysis from the previous agent,                                                   │
│          retrieve information to answer this question: What are the key benefits of Python for data science?    │
│                                                                                                                 │
│          Then synthesize a comprehensive answer with:                                                           │
│          1. Direct answer to the question                                                                       │
│          2. Supporting details from retrieved sources                                                           │
│          3. Proper source attribution and citations                                                             │
│          4. Confidence level assessment                                                                         │
│  Agent: Information Retriever and Answer Synthesizer                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-23 15:24:14,798 - __main__ - INFO - [System] Query completed



✅ ANSWER GENERATED
Direct answer (summary)
- Python’s key benefits for data science are: easy readability and rapid prototyping; a rich, well‑maintained ecosystem of purpose-built libraries (NumPy, pandas, scikit‑learn, etc.); excellent interactive and notebook tooling (Jupyter) for exploration and reproducibility; strong visualization options; good performance for array/math work via optimized native libraries; easy deployment and integration with production systems; and a large, active community and distribution tooling (Anaconda, PyData) that simplifies package, environment, and workflow management. These features together make Python highly productive for the full data‑science lifecycle: data ingestion → cleaning → analysis → modeling → visualization → deployment. (See supporting evidence and sources below.)

Supporting details and evidence (by benefit)

1) Readability, ease of learning, and rapid prototyping
- Python’s simple, readable syntax reduces cognitive overhead when writi

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e097df75-afa8-4cb3-ac58-cf8e10c58e1b                                                                       │
│  Final Output: Direct answer (summary)                                                                          │
│  - Python’s key benefits for data science are: easy readability and rapid prototyping; a rich, well‑maintained  │
│  ecosystem of purpose-built libraries (NumPy, pandas, scikit‑learn, etc.); excellent interactive and notebook   │
│  tooling (Jupyter) for exploration and reproducibility; strong visualization options; good performance for      │
│  array/math work via optimized native libraries; easy deployment and integration with production systems; and   │
│  a large, active community and distribution tooling (Anaconda, PyData) that simplifies package, environment,    │
│  and workflow management. These features together make Python highly productive for the full data‑science       │
│  lifecycle: data ingestion → cleaning → analysis → modeling → visualization → deployment. (See supporting       │
│  evidence and sources below.)                                                                                   │
│                                                                                                                 │
│  Supporting details and evidence (by benefit)                                                                   │
│                                                                                                                 │
│  1) Readability, ease of learning, and rapid prototyping                                                        │
│  - Python’s simple, readable syntax reduces cognitive overhead when writing data‑manipulation and modeling      │
│  code and speeds iteration and prototyping. This is a core reason it is widely taught and used in data science  │
│  workflows. (General language rationale; see community/ecosystem sources below for adoption context.)           │
│    Source: PyData / community references [6].                                                                   │
│                                                                                                                 │
│  2) Rich ecosystem of libraries for numeric computing and data frames                                           │
│  - NumPy provides the fundamental n‑dimensional array object and vectorized operations implemented in           │
│  optimized C, which gives numerical speed and is the foundation for most scientific computing in Python [1].    │
│    Source: NumPy “What is NumPy?” [1].                                                                          │
│  - pandas builds on NumPy to provide DataFrame objects and high‑level I/O and data‑cleaning APIs that make      │
│  tabular data manipulation concise and expressive (grouping, joins, time series, missing‑data handling) [2].    │
│    Source: pandas documentation [2].                                                                            │
│                                                                                                                 │
│  3) Mature, easy‑to‑use machine‑learning libraries                                                              │
│  - scikit‑learn offers a consistent, well‑documented API for common ML tasks (preprocessing,                    │
│  supervised/unsupervised models, model selection) that is production‑ready for many classical ML workflows      │
│  [3]. This lowers the barrier to try many algorithms q

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯